# Notebook 03 — Statcast Data Pull (Baseball Savant)

Source: `baseballsavant.mlb.com` custom leaderboard CSV export  
Rate limit: 1 s between calls, browser-like User-Agent  

This notebook pulls two Statcast leaderboards for the 2025 season:

1. Hitting — xBA, xSLG, xwOBA, xOBP, xISO, avg_swing_speed, fast_swing_rate, blasts_contact, blasts_swing, squared_up_contact, squared_up_swing, avg_swing_length, swords, attack_angle, attack_direction, ideal_angle_rate, vertical_swing_path, exit_velocity_avg  launch_angle_avg, sweet_spot_percent, barrel_batted_rate
2. Pitching — xERA, xBA, xSLG, xwOBA, xOBP, xISO, wOBA, k_percent, bb_percent, whiff_percent, chase_percent, exit_velocity_avg  sweet_spot_percent, barrel_batted_rate, hard_hit_percent, groundballs_percent  

Join key is `mlbam_id` (the `player_id` column in Savant's CSV).

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))

from scrapers.baseball_savant import get_statcast_hitting, get_statcast_pitching
import pandas as pd

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 140)

## 1. Statcast Hitting Leaderboard
### What each metric measures and why it's more predictive

| Statcast Metric | Group | What it is |
|---|---|---|
| **xBA** (Expected Batting Average) | Expected Outcomes | Built from exit velocity and launch angle on every batted ball. Removes BABIP noise from batting average. Year-to-year correlation is ~0.75 vs. ~0.60 for AVG. |
| **xSLG** (Expected Slugging) | Expected Outcomes | Same EV/LA model applied to extra-base power. Reflects true contact quality regardless of park effects or whether a fly ball carried. |
| **xwOBA** (Expected Weighted On-Base Average) | Expected Outcomes | Combines contact quality with walk/strikeout rates on a run-value scale. More stable than actual wOBA because defense and park are removed from batted-ball outcomes. |
| **xOBP** (Expected On-Base Percentage) | Expected Outcomes | Applies the expected model to hit outcomes while keeping real BB/HBP. Removes BABIP luck from the hits component. |
| **xISO** (Expected Isolated Power) | Expected Outcomes | Expected extra-base power per at-bat. Mirrors traditional ISO but uses the EV/LA model to separate true power from park and luck. |
| **Avg Swing Speed** | Bat Tracking | Average bat speed measured 6 inches from the barrel across competitive swings. Faster swings create more potential exit velocity. |
| **Fast Swing Rate** | Bat Tracking | Percentage of swings at or above 75 mph bat speed. Shows how often a hitter generates elite swing speed. |
| **Blasts (Contact %)** | Bat Tracking | Percentage of contact events classified as a "blast": a swing that is both fast and squared up. Strong single-metric for hard contact production. |
| **Blasts (Swing %)** | Bat Tracking | Percentage of all swings that produce blast contact. Lower than blasts_contact because it includes swings that miss entirely. |
| **Squared-Up Contact %** | Bat Tracking | Percentage of contact where the batter reached close to the maximum possible exit velocity given bat speed and pitch speed. Tracks contact precision. |
| **Squared-Up Swing %** | Bat Tracking | Percentage of all swings (including misses) that result in squared-up contact. Paired with whiff rate to gauge overall swing efficiency. |
| **Avg Swing Length** | Bat Tracking | Average distance the barrel travels from start to contact in XYZ space (feet). Shorter paths mean quicker swings and less vulnerability to inside pitches. |
| **Swords** | Bat Tracking | Count of swings resulting in an extreme miss. The batter is beaten badly enough that the swing looks awkward. High sword counts signal vulnerability to quality stuff. |
| **Attack Angle** | Bat Tracking | Vertical angle of the bat at contact (degrees). Positive = upward swing plane. Optimal range (~5–15°) aligns with average pitch descent to maximize contact area. |
| **Attack Direction** | Bat Tracking | Horizontal swing direction at contact relative to home plate. Reflects pull tendency vs. opposite-field approach in the swing mechanics, independent of batted-ball outcomes. |
| **Ideal Angle Rate** | Bat Tracking | Percentage of swings where attack angle falls in the optimal launch window. Higher rates mean the swing plane consistently matches pitch trajectory. |
| **Vertical Swing Path** | Bat Tracking | Steepness of the swing plane from start to contact. Works alongside attack angle to describe the full 3D geometry of the swing. |
| **Exit Velocity (Avg)** | Batted Ball Outcomes | Average speed off the bat across all batted balls. Direct product of swing speed and contact quality. Strong predictor of wOBA and HR rate. |
| **Launch Angle (Avg)** | Batted Ball Outcomes | Average vertical angle of the batted ball. The 10–30° range produces the most run value. Explains why two hitters with the same EV can have very different results. |
| **Sweet Spot %** | Batted Ball Outcomes | Percentage of batted balls hit between 8° and 32° launch angle. Balls in this window have the highest hit probability and slugging potential. |
| **Barrel %** | Batted Ball Outcomes | Percentage of batted balls meeting the optimal EV/LA threshold (≥98 mph, 26–30°). Barrels carry a ~1.000 xSLG and are the most consistent power indicator year over year.|

In [ ]:
hitting = get_statcast_hitting(2025)
print(f"Shape: {hitting.shape}")
print(f"\nDtypes:\n{hitting.dtypes}")
print(f"\nNull rates:\n{hitting.isnull().mean().round(4)}")
print(f"\nSample (10 rows):")
hitting.head(10)

## 2. Statcast Pitching Leaderboard
### What each metric measures and why it's more predictive

| Statcast Metric | Group | What it is |
|---|---|---|
| **xERA** (Expected ERA) | Expected Outcomes | ERA built from contact quality allowed (EV + LA model) rather than actual runs. Removes defense, sequencing, and strand-rate luck, making it more park- and defense-neutral than standard ERA estimators. |
| **xBA** (Expected Batting Average Allowed) | Expected Outcomes | Expected hit rate against, based on the EV/LA of every batted ball the pitcher allowed. Removes BABIP noise. A pitcher with low xBA but high BA is likely benefiting from good defense or sequencing. |
| **xSLG** (Expected Slugging Allowed) | Expected Outcomes | EV/LA model applied to extra-base contact allowed. Reflects true power damage regardless of park or whether fly balls happened to carry. |
| **xwOBA** (Expected wOBA Allowed) | Expected Outcomes | Combines contact quality with real strikeouts and walks on a run-value scale. More stable than actual wOBA allowed because defense is removed from batted-ball outcomes. Widely considered the strongest single contact-quality read on a pitcher. |
| **xOBP** (Expected OBP Allowed) | Expected Outcomes | Applies the expected model to hit outcomes while keeping real BB/HBP allowed. Distinguishes between high OBP allowed driven by walks versus hit luck. |
| **xISO** (Expected ISO Allowed) | Expected Outcomes | Expected extra-base power allowed per at-bat. Separates true power suppression from park effects and HR/FB luck. |
| **wOBA** (Weighted On-Base Average Allowed) | Expected Outcomes | Actual run-value-weighted on-base average allowed. Included alongside xwOBA so the gap between the two can be tracked as a luck indicator. |
| **K%** (Strikeout Rate) | Plate Discipline | Strikeouts per plate appearance. Defense plays no role in strikeouts, which is why K% holds up better than most pitching stats year over year and is a strong predictor of ERA estimators. |
| **BB%** (Walk Rate) | Plate Discipline | Walks per plate appearance. Free baserunners with no defense involved. Combined with K% into K-BB%, it gives the clearest read on a pitcher's command. |
| **Whiff %** | Plate Discipline | Swings and misses divided by total swings. Tracks swing-and-miss ability on contact attempts specifically, making it more granular than SwStr%, which includes pitches batters never offered at. |
| **Chase %** (OZ Swing %) | Plate Discipline | Percentage of pitches outside the zone that batters swing at. Higher rates mean hitters are making poor decisions, usually a function of pitch quality and location at the edges. |
| **Exit Velocity (Avg Allowed)** | Batted Ball Outcomes | Average EV on all batted balls against. Shows how hard the pitcher is being hit. Less park-dependent than ERA or HR rate. |
| **Sweet Spot % (Allowed)** | Batted Ball Outcomes | Percentage of batted balls against landing in the 8–32° launch angle window. Balls in this range have the highest hit probability and slugging potential. Keeping this rate low means the pitcher is generating weak or mis-hit contact. |
| **Barrel % (Allowed)** | Batted Ball Outcomes | Percentage of batted balls against meeting the barrel threshold (≥98 mph, 26–30° LA). Barrels carry ~1.000 xSLG against and this rate is one of the more consistent damage-allowed metrics year over year. |
| **Hard-Hit % (Allowed)** | Batted Ball Outcomes | Percentage of batted balls against at ≥95 mph exit velocity. Broader than barrel rate, it picks up overall contact damage including line drives that don't meet the barrel angle threshold. |
| **GB% (Allowed)** | Batted Ball Outcomes | Ground ball rate on batted balls against. Ground balls have the lowest xSLG of any batted ball type and can't leave the park. High GB% pitchers are more defense-dependent but limit big innings. |

In [ ]:
pitching = get_statcast_pitching(2025)
print(f"Shape: {pitching.shape}")
print(f"\nDtypes:\n{pitching.dtypes}")
print(f"\nNull rates:\n{pitching.isnull().mean().round(4)}")
print(f"\nSample (10 rows):")
pitching.head(10)

## 3. Null Rate Summary (both DataFrames)

In [ ]:
null_summary = pd.DataFrame({
    "hitting": hitting.isnull().mean().round(4),
    "pitching": pitching.isnull().mean().round(4),
}).fillna("—")
print("=== Null Rate Summary ===")
null_summary

## 4. Save to Parquet


In [ ]:
data_dir = os.path.join("..", "data")
os.makedirs(data_dir, exist_ok=True)

hitting.to_parquet(os.path.join(data_dir, "raw_statcast_hitting.parquet"), index=False)
pitching.to_parquet(os.path.join(data_dir, "raw_statcast_pitching.parquet"), index=False)

print("Saved:")
for f in ["raw_statcast_hitting.parquet", "raw_statcast_pitching.parquet"]:
    path = os.path.join(data_dir, f)
    size_kb = os.path.getsize(path) / 1024
    print(f"  {f} — {size_kb:.1f} KB")